# Phase 5 — Audit des variables et suppression des fuites de données

## Objectifs

- Identifier les informations utilisées par le premier modèle.
- Déterminer qui renseigne chaque information et à quel moment.
- Identifier les variables qui révèlent déjà la cible ou qui ne sont pas
  disponibles lors de l'arrivée d'un nouveau signalement.
- Entraîner un nouveau modèle sans ces variables.
- Comparer les résultats du premier modèle et du modèle sans fuite.

## Imports

In [21]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## Chemins et noms des colonnes

In [22]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## Charger les lignes structurées

In [23]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes chargées : {len(df)}")
print(f"Lignes isolées : {len(lignes_problemes)}")

Lignes chargées : 88679
Lignes isolées : 196


## Refaire les conversions

In [24]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

## Recréer la cible `is_hoax`

In [25]:
MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

pattern_canular = "|".join(
    re.escape(mot) for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

print(df["is_hoax"].value_counts())

is_hoax
0    87810
1      869
Name: count, dtype: int64


## Construire les variables temporelles

In [26]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["posted_year"] = df["date_posted"].dt.year

## Tableau d’audit des variables

In [27]:
audit_variables = pd.DataFrame(
    [
        {
            "colonne": "comments",
            "qui_renseigne": "Témoin",
            "moment": "Lors de la déclaration",
            "connait_deja_la_cible": "Oui : la cible is_hoax est construite à partir de ce texte",
            "decision": "Supprimée",
        },
        {
            "colonne": "city",
            "qui_renseigne": "Témoin",
            "moment": "Lors de la déclaration",
            "connait_deja_la_cible": "Non",
            "decision": "Conservée",
        },
        {
            "colonne": "state",
            "qui_renseigne": "Témoin ou système de géocodage",
            "moment": "Lors de la déclaration ou juste après",
            "connait_deja_la_cible": "Non directement",
            "decision": "Conservée",
        },
        {
            "colonne": "country",
            "qui_renseigne": "Témoin ou système de géocodage",
            "moment": "Lors de la déclaration ou juste après",
            "connait_deja_la_cible": "Non directement",
            "decision": "Conservée",
        },
        {
            "colonne": "shape",
            "qui_renseigne": "Témoin",
            "moment": "Lors de la déclaration",
            "connait_deja_la_cible": "Non",
            "decision": "Conservée",
        },
        {
            "colonne": "duration_seconds",
            "qui_renseigne": "Témoin ou système de traitement",
            "moment": "Lors de la déclaration ou juste après",
            "connait_deja_la_cible": "Non directement",
            "decision": "Conservée",
        },
        {
            "colonne": "latitude",
            "qui_renseigne": "Capteur ou système de géocodage",
            "moment": "Après localisation du signalement",
            "connait_deja_la_cible": "Non directement",
            "decision": "Conservée si disponible immédiatement",
        },
        {
            "colonne": "longitude",
            "qui_renseigne": "Capteur ou système de géocodage",
            "moment": "Après localisation du signalement",
            "connait_deja_la_cible": "Non directement",
            "decision": "Conservée si disponible immédiatement",
        },
        {
            "colonne": "datetime",
            "qui_renseigne": "Témoin",
            "moment": "Lors de la déclaration",
            "connait_deja_la_cible": "Non",
            "decision": "Conservée sous forme année, mois et heure",
        },
        {
            "colonne": "date_posted",
            "qui_renseigne": "Service de traitement",
            "moment": "Après réception et traitement",
            "connait_deja_la_cible": "Oui ou non disponible au moment de la prédiction",
            "decision": "Supprimée",
        },
    ]
)

audit_variables

,colonne,qui_renseigne,moment,connait_deja_la_cible,decision
0,comments,Témoin,Lors de la déclaration,Oui : la cible is_hoax est construite à partir...,Supprimée
1,city,Témoin,Lors de la déclaration,Non,Conservée
2,state,Témoin ou système de géocodage,Lors de la déclaration ou juste après,Non directement,Conservée
3,country,Témoin ou système de géocodage,Lors de la déclaration ou juste après,Non directement,Conservée
4,shape,Témoin,Lors de la déclaration,Non,Conservée
5,duration_seconds,Témoin ou système de traitement,Lors de la déclaration ou juste après,Non directement,Conservée
6,latitude,Capteur ou système de géocodage,Après localisation du signalement,Non directement,Conservée si disponible immédiatement
7,longitude,Capteur ou système de géocodage,Après localisation du signalement,Non directement,Conservée si disponible immédiatement
8,datetime,Témoin,Lors de la déclaration,Non,"Conservée sous forme année, mois et heure"
9,date_posted,Service de traitement,Après réception et traitement,Oui ou non disponible au moment de la prédiction,Supprimée


## Exporter le tableau d’audit

In [28]:
audit_variables.to_csv(
    OUTPUT_DIR / "audit_variables_phase5.csv",
    index=False,
)

## Créer le texte sans fuite

In [29]:
df["text_features_without_leakage"] = (
    "city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

df[
    [
        "text_features_without_leakage",
        "is_hoax",
    ]
].head()

,text_features_without_leakage,is_hoax
0,city san marcos state tx country us shape cyli...,0
1,city lackland afb state tx country shape light,0
2,city chester (uk/england) state country gb sh...,0
3,city edna state tx country us shape circle,0
4,city kaneohe state hi country us shape light,0


## Construire `x` et `y`

In [30]:
features_numeriques_sans_fuite = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
]

features_modele_sans_fuite = [
    "text_features_without_leakage",
] + features_numeriques_sans_fuite

X_sans_fuite = df[features_modele_sans_fuite].copy()
y = df["is_hoax"].copy()

print("Dimensions de X :", X_sans_fuite.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (88679, 7)
Dimensions de y : (88679,)


## Créer exactement le même train/test

In [31]:
indices_train, indices_test = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train_sans_fuite = X_sans_fuite.loc[indices_train]
X_test_sans_fuite = X_sans_fuite.loc[indices_test]

y_train = y.loc[indices_train]
y_test = y.loc[indices_test]

print("Taille entraînement :", X_train_sans_fuite.shape)
print("Taille test :", X_test_sans_fuite.shape)

print(f"Part de canulars train : {y_train.mean():.2%}")
print(f"Part de canulars test : {y_test.mean():.2%}")

Taille entraînement : (70943, 7)
Taille test : (17736, 7)
Part de canulars train : 0.98%
Part de canulars test : 0.98%


## Prétraitement sans fuite

In [32]:
preprocessing_sans_fuite = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=10_000,
                ngram_range=(1, 2),
            ),
            "text_features_without_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            features_numeriques_sans_fuite,
        ),
    ]
)

## Créer le modèle sans fuite

In [33]:
modele_sans_fuite = Pipeline(
    steps=[
        ("preprocessing", preprocessing_sans_fuite),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

## Entraîner

In [34]:
modele_sans_fuite.fit(
    X_train_sans_fuite,
    y_train,
)

print("Entraînement du modèle sans fuite terminé.")

Entraînement du modèle sans fuite terminé.


## Prédire

In [35]:
y_pred_sans_fuite = modele_sans_fuite.predict(
    X_test_sans_fuite
)

pd.Series(y_pred_sans_fuite).value_counts()

0    13351
1     4385
Name: count, dtype: int64

## Calculer les métriques

In [36]:
precision_sans_fuite = precision_score(
    y_test,
    y_pred_sans_fuite,
    zero_division=0,
)

recall_sans_fuite = recall_score(
    y_test,
    y_pred_sans_fuite,
    zero_division=0,
)

accuracy_sans_fuite = accuracy_score(
    y_test,
    y_pred_sans_fuite,
)

print(f"Precision sans fuite : {precision_sans_fuite:.2%}")
print(f"Recall sans fuite : {recall_sans_fuite:.2%}")
print(f"Accuracy sans fuite : {accuracy_sans_fuite:.2%}")

Precision sans fuite : 1.69%
Recall sans fuite : 42.53%
Accuracy sans fuite : 75.13%


## Matrice de confusion

In [37]:
matrice_sans_fuite = confusion_matrix(
    y_test,
    y_pred_sans_fuite,
)

df_matrice_sans_fuite = pd.DataFrame(
    matrice_sans_fuite,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

df_matrice_sans_fuite

,Prédit : non-canular,Prédit : canular
Réel : non-canular,13251,4311
Réel : canular,100,74


## Rapport de classification

In [38]:
print(
    classification_report(
        y_test,
        y_pred_sans_fuite,
        target_names=[
            "non-canular",
            "canular",
        ],
        zero_division=0,
    )
)

              precision    recall  f1-score   support

 non-canular       0.99      0.75      0.86     17562
     canular       0.02      0.43      0.03       174

    accuracy                           0.75     17736
   macro avg       0.50      0.59      0.44     17736
weighted avg       0.98      0.75      0.85     17736



## Recréer les métriques du modèle initial

In [39]:
df["text_features_with_leakage"] = (
    "comment " + df["comments"].fillna("").astype(str)
    + " city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

features_numeriques_avec_fuite = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
    "posted_year",
]

features_avec_fuite = [
    "text_features_with_leakage",
] + features_numeriques_avec_fuite

X_avec_fuite = df[features_avec_fuite].copy()

preprocessing_avec_fuite = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=30_000,
                ngram_range=(1, 2),
            ),
            "text_features_with_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            features_numeriques_avec_fuite,
        ),
    ]
)

modele_avec_fuite = Pipeline(
    steps=[
        ("preprocessing", preprocessing_avec_fuite),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

modele_avec_fuite.fit(
    X_avec_fuite.loc[indices_train],
    y.loc[indices_train],
)

y_pred_avec_fuite = modele_avec_fuite.predict(
    X_avec_fuite.loc[indices_test]
)

precision_avec_fuite = precision_score(
    y.loc[indices_test],
    y_pred_avec_fuite,
    zero_division=0,
)

recall_avec_fuite = recall_score(
    y.loc[indices_test],
    y_pred_avec_fuite,
    zero_division=0,
)

accuracy_avec_fuite = accuracy_score(
    y.loc[indices_test],
    y_pred_avec_fuite,
)

print(f"Precision avec fuite : {precision_avec_fuite:.2%}")
print(f"Recall avec fuite : {recall_avec_fuite:.2%}")
print(f"Accuracy avec fuite : {accuracy_avec_fuite:.2%}")

c:\Users\serge\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Precision avec fuite : 1.65%
Recall avec fuite : 64.37%
Accuracy avec fuite : 62.07%


## Comparer avant et après

In [40]:
resultats_phase5 = pd.DataFrame(
    [
        {
            "version_modele": "Avec fuite",
            "precision": precision_avec_fuite,
            "recall": recall_avec_fuite,
            "accuracy": accuracy_avec_fuite,
        },
        {
            "version_modele": "Sans fuite",
            "precision": precision_sans_fuite,
            "recall": recall_sans_fuite,
            "accuracy": accuracy_sans_fuite,
        },
    ]
)

resultats_phase5

,version_modele,precision,recall,accuracy
0,Avec fuite,0.016524,0.643678,0.620659
1,Sans fuite,0.016876,0.425287,0.751297


## Exporter les résultats

In [41]:
resultats_phase5.to_csv(
    OUTPUT_DIR / "resultats_phase5_avant_apres_fuite.csv",
    index=False,
)

df_matrice_sans_fuite.to_csv(
    OUTPUT_DIR / "matrice_confusion_modele_sans_fuite.csv",
    index=True,
)

print("Fichiers exportés.")

Fichiers exportés.
